In [3]:
import torch

# GPT2 using Decoder Only

## Scaled dot-product attention

Attention(Q, K, V) = softmax($\frac{QK^T}{\sqrt{d_k}}V$)

In [4]:
import torch.nn as nn

In [5]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_model, num_heads, d_k=None, d_v=None):
    super().__init__()
    self.num_heads = num_heads
    self.d_k = d_k or d_model // num_heads
    self.d_v = d_v or d_model // num_heads
    self.W_q = nn.Linear(d_model, self.d_k * num_heads)
    self.W_k = nn.Linear(d_model, self.d_k * num_heads)
    self.W_v = nn.Linear(d_model, self.d_v * num_heads)
    self.W_o = nn.Linear(self.d_v * num_heads, d_model)

  def forward(self, query, key, value, is_mask=None):
    q = self.W_q(query)
    k = self.W_k(key)
    v = self.W_v(value)

    # divide into heads
    B, N_q, _ = q.size()
    _, N_k, _ = k.size()
    q = q.reshape(B, N_q, self.num_heads, self.d_k).transpose(1, 2)
    k = k.reshape(B, N_k, self.num_heads, self.d_k).transpose(1, 2)
    v = v.reshape(B, N_k, self.num_heads, self.d_v).transpose(1, 2)

    scores = q @ k.transpose(-2, -1) / self.d_k ** 0.5
    if is_mask is not None:
      mask = torch.triu(torch.ones(N_q, N_k, device=scores.device), diagonal=1).bool()
      scores = scores.masked_fill(mask, -torch.inf)

    scores = torch.softmax(scores, dim=-1)
    final_vecs = scores @ v

    # concat
    final_vecs = final_vecs.transpose(1, 2).reshape(B, N_q, -1)
    return self.W_o(final_vecs)

In [6]:
mha = MultiHeadAttention(768, 6)

In [7]:
q = torch.randn(2, 64, 768)
k = v = torch.randn(2, 48, 768)
final_vecs = mha(q, k, v, True)

In [8]:
final_vecs.shape

torch.Size([2, 64, 768])

In [9]:
class FNN(nn.Module):
  def __init__(self, d_model, d_ff):
    super().__init__()
    self.linear_layer = nn.Linear(d_model, d_ff)
    self.relu = nn.ReLU()
    self.linear_layer2 = nn.Linear(d_ff, d_model)

  def forward(self, x):
    return self.linear_layer2(self.relu(self.linear_layer(x)))

In [10]:
x = torch.randn(2, 64, 768)
mha = MultiHeadAttention(768, 6)
fnn = FNN(768, 768 * 2)
result = fnn(mha(x, x, x))
result.shape

torch.Size([2, 64, 768])

In [11]:
class MyGPT2Block(nn.Module):
  def __init__(self, d_model, num_heads, d_ff):
    super().__init__()
    self.masked_mha = MultiHeadAttention(d_model, num_heads)
    self.fnn = FNN(d_model, d_ff)
    self.ln1 = nn.LayerNorm(d_model)
    self.ln2 = nn.LayerNorm(d_model)

  def forward(self, x):
    old_x = x
    x_norm = self.ln1(x)
    x = old_x + self.masked_mha(x_norm, x_norm, x_norm, True)
    old_x = x
    x_norm = self.ln2(x)
    x = old_x + self.fnn(x_norm)
    return x

In [12]:
my_gpt2_block = MyGPT2Block(768, 6, 768 * 2)
result = my_gpt2_block(x)
result = my_gpt2_block(result)
result = my_gpt2_block(result)
result = my_gpt2_block(result)
result = my_gpt2_block(result)
result = my_gpt2_block(result)
result.shape

torch.Size([2, 64, 768])

In [13]:
embedding = nn.Embedding(50012, 768)
all(embedding(torch.tensor(0)) == embedding.weight[0])

True

In [14]:
class MyGPT2(nn.Module):
  def __init__(self, d_model, num_heads, d_ff, num_layers, vocab_size, max_seq_len):
    super().__init__()
    self.embed_layer = nn.Embedding(vocab_size, d_model)
    self.pos_embed_layer = nn.Embedding(max_seq_len, d_model)
    self.layers = nn.ModuleList([
      MyGPT2Block(d_model, num_heads, d_ff) for _ in range(num_layers)
    ])
    self.ln_f = nn.LayerNorm(d_model)
    self.linear = nn.Linear(d_model, vocab_size)

  def forward(self, x):
    seq_len = x.size(1)
    positions = torch.arange(0, seq_len, device=x.device).unsqueeze(0)
    x = self.embed_layer(x) + self.pos_embed_layer(positions)
    for layer in self.layers:
      x = layer(x)
    x = self.ln_f(x)
    logits = self.linear(x)
    return logits

In [15]:
my_gpt2 = MyGPT2(768, 6, 768 * 2, 12, 50257, 512)
x_test = torch.randint(10, (2, 128))
result = my_gpt2(x_test)
result.shape

torch.Size([2, 128, 50257])

# Tokenizer

In [16]:
from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

In [17]:
ids = tokenizer.encode("hello, world")

In [18]:
ids

[31373, 11, 995]

In [19]:
tokenizer.convert_ids_to_tokens(ids)

['hello', ',', 'Ġworld']

# Prepare dataset

In [20]:
def get_batch(token_ids, block_size, batch_size):
  """Get random sample for token_ids"""
  start_idx = torch.randint(len(token_ids) - block_size, (batch_size,))
  x = torch.stack([torch.tensor(token_ids[i:i+block_size]) for i in start_idx])
  y = torch.stack([torch.tensor(token_ids[i+1:i+1+block_size]) for i in start_idx])
  return x, y

In [21]:
data = "My name is John. What is your name?"
token_ids = tokenizer.encode(data)
token_ids

[3666, 1438, 318, 1757, 13, 1867, 318, 534, 1438, 30]

In [22]:
data, label = get_batch(token_ids, 4, 8)

In [23]:
for (x, y) in zip(data, label):
  print(tokenizer.decode(x))
  print(tokenizer.decode(y))
  print("----------------")

 John. What is
. What is your
----------------
 is John. What
 John. What is
----------------
. What is your
 What is your name
----------------
. What is your
 What is your name
----------------
 name is John.
 is John. What
----------------
 John. What is
. What is your
----------------
 name is John.
 is John. What
----------------
 name is John.
 is John. What
----------------


In [24]:
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

In [25]:
class TextDataset(Dataset):
  def __init__(self, token_ids, block_size):
    self.token_ids = torch.tensor(token_ids, dtype=torch.long)
    self.block_size = block_size

  def __len__(self):
    return len(self.token_ids) - self.block_size

  def  __getitem__(self, idx):
    x = self.token_ids[idx : idx + self.block_size]
    y = self.token_ids[idx + 1 : idx + self.block_size +1]
    return x, y

In [26]:
data = "Lifelong learning has become one of the most important habits a person can develop because the world is changing faster than ever before, and the knowledge or skills that are useful today may not be enough to meet the challenges of tomorrow. In the past, many people could complete their formal education, enter a profession, and expect to use roughly the same knowledge for most of their working lives, but modern technology, globalization, scientific discoveries, and social change have transformed the way people live and work. New tools appear constantly, industries evolve, and information can spread around the world within seconds, which means that individuals need to remain curious, flexible, and willing to learn throughout their lives. Lifelong learning does not only mean attending school or earning additional degrees; it includes reading books, taking online courses, learning from colleagues, practicing new skills, exploring unfamiliar subjects, asking questions, reflecting on mistakes, and gaining knowledge through daily experiences. One of the greatest benefits of lifelong learning is that it helps people adapt to change with greater confidence. When a workplace introduces new software, when a profession begins using new methods, or when a person decides to change careers, the ability to learn quickly can reduce fear and create new opportunities. A person who is accustomed to learning is more likely to see change as a challenge that can be understood rather than as a threat that must be avoided. Continuous learning can also improve career development because employers often value workers who are willing to update their skills, solve problems, communicate effectively, and take initiative. Even when a particular skill is not directly related to a person’s job, learning can strengthen abilities such as critical thinking, creativity, organization, and decision-making, which are useful in almost every profession. Beyond employment, lifelong learning can make personal life richer and more meaningful. Learning a new language can help someone communicate with people from different cultures, studying history can provide a deeper understanding of current events, learning to cook can improve independence and health, and practicing music, art, gardening, or another hobby can provide enjoyment and a sense of achievement. Learning also encourages people to remain mentally active because it requires attention, memory, reasoning, and imagination. Perhaps even more importantly, lifelong learning can teach humility by reminding people that no matter how much they know, there is always more to discover. This attitude is especially valuable in an age when people are surrounded by enormous amounts of information, some of which is inaccurate, misleading, or incomplete. A lifelong learner does not simply accept every claim but develops the habit of checking evidence, comparing sources, considering different perspectives, and changing an opinion when better information becomes available. These habits support responsible decision-making in areas such as health, finance, relationships, education, and public life. Lifelong learning can also strengthen relationships because learning often requires listening to other people and trying to understand experiences that are different from one’s own. When people approach conversations with curiosity rather than certainty, they may become more patient, respectful, and open-minded. This does not mean that they must agree with everyone, but it means they can disagree while still seeking to understand the reasons behind another person’s viewpoint. Another important aspect of lifelong learning is the ability to learn from failure. Many people are afraid of mistakes because they associate them with embarrassment or weakness, yet mistakes often provide some of the most useful lessons. A person who learns continuously can view failure as information: something did not work, so the next step is to understand why, make adjustments, and try again. This mindset is important for entrepreneurs, students, professionals, artists, athletes, and anyone trying to improve at a difficult task. However, lifelong learning requires discipline because curiosity alone is not always enough. People are busy, distractions are everywhere, and it is easy to postpone learning until there is more free time. A practical approach is to make learning a regular part of everyday life by setting realistic goals, such as reading for twenty minutes each day, completing one lesson from an online course each week, keeping notes about new ideas, or practicing a skill consistently. Small efforts may seem insignificant at first, but they can produce remarkable results when repeated over months and years. It is also helpful to choose learning goals that are meaningful rather than following every new trend. People learn more effectively when they understand why a subject matters to them and how they hope to use it. At the same time, exploring topics simply for curiosity can be valuable because unexpected connections between different fields often lead to creativity and innovation. Someone who studies both technology and psychology, for example, may understand not only how a digital product works but also how people interact with it. A person interested in business who also studies communication may become better at leadership, negotiation, and teamwork. In this way, learning across different subjects can help people develop a broader view of problems and discover solutions that might not be obvious from a single perspective. Society also benefits when people continue learning because educated, curious citizens are better prepared to understand complex issues, participate in their communities, and contribute useful ideas. Communities become stronger when people are willing to share knowledge, teach one another, and remain open to better ways of doing things. Ultimately, lifelong learning is not about knowing everything, which is impossible, but about maintaining the willingness to grow. It is a commitment to curiosity, adaptability, reflection, and improvement. The most successful learner is not necessarily the person who memorizes the most facts, but the person who knows how to ask useful questions, find reliable information, apply knowledge thoughtfully, and continue developing when circumstances change. In a world where change is unavoidable, the ability to keep learning provides a form of lasting security because even when particular technologies, jobs, or situations become outdated, the capacity to learn remains useful. By making learning a lifelong habit, people can prepare themselves for uncertainty, discover new interests, build stronger careers, make wiser decisions, connect with others, and experience the satisfaction that comes from continual growth. For these reasons, lifelong learning should not be viewed as something that ends when a person leaves school; it is an ongoing process that can shape an entire life, helping individuals remain capable, curious, and engaged no matter how much the world changes."

In [27]:
token_ids = tokenizer.encode(data)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1242 > 1024). Running this sequence through the model will result in indexing errors


In [28]:
text_ds = TextDataset(token_ids, 128)

In [29]:
text_ds

In [30]:
a = torch.randn(5)
a.shape

torch.Size([5])

In [31]:
x, y = text_ds[2]

In [32]:
all(x[1:] == y[:-1])

True

In [33]:
loader = DataLoader(text_ds, batch_size=32, shuffle=True)

In [34]:
loader

In [35]:
x_test = None
y_test = None
for x, y in loader:
  x_test = x
  y_test = y
  break

In [36]:
x_test.shape
y_test.shape

torch.Size([32, 128])

In [37]:
logits = my_gpt2(x_test)

# Loss fn

In [38]:
import torch.nn.functional as F
import torch.nn as nn

In [39]:
y.shape

torch.Size([32, 128])

In [40]:
logits.shape

torch.Size([32, 128, 50257])

In [41]:
loss = F.cross_entropy(
    logits.view(-1, 50257),
    y.view(-1)
)

In [42]:
loss

tensor(11.0485, grad_fn=<NllLossBackward0>)

In [43]:
optimizer = torch.optim.Adam(my_gpt2.parameters(), lr=1e-3)

In [44]:
from tqdm import tqdm

In [45]:
device = "cuda"

In [47]:
EPOCH_NUM = 5
my_gpt2.to(device)
for i in range(EPOCH_NUM):
  train_loss = 0.0
  for x, y in tqdm(loader):
    x = x.to(device)
    y = y.to(device)
    optimizer.zero_grad()
    logits = my_gpt2(x)
    loss = F.cross_entropy(
        logits.view(-1, 50257),
        y.view(-1)
    )
    train_loss += loss.item()
    loss.backward()
    optimizer.step()
  train_loss /= len(loader)
  print(f"Train Loss: {train_loss:.4f}")

100%|██████████| 35/35 [00:26<00:00,  1.32it/s]


Train Loss: 5.7499


100%|██████████| 35/35 [00:27<00:00,  1.28it/s]


Train Loss: 3.5239


100%|██████████| 35/35 [00:28<00:00,  1.23it/s]


Train Loss: 1.2439


100%|██████████| 35/35 [00:30<00:00,  1.16it/s]


Train Loss: 0.3589


100%|██████████| 35/35 [00:31<00:00,  1.11it/s]

Train Loss: 0.0879


In [48]:
input = "Lifelong learning"
input_ids = torch.tensor(tokenizer.encode(input), dtype=torch.long)

In [52]:
input_ids = input_ids.unsqueeze(0)

In [59]:
logits = my_gpt2(input_ids.to(device))

In [60]:
logits.shape

torch.Size([1, 4, 50257])

In [61]:
next_token_logits = logits[:, -1, :]

In [62]:
next_token_id = torch.argmax(next_token_logits, dim=-1)

In [65]:
tokenizer.decode(next_token_id.item())

' does'

In [74]:
input_ids = tokenizer.encode("It is also")
for _ in range(20):
  input_ids_tensor = torch.tensor(input_ids).unsqueeze(0).to(device)
  logits = my_gpt2(input_ids_tensor)
  next_token_logits = logits[:, -1, :]
  next_token_id = torch.argmax(next_token_logits, dim=-1)
  input_ids.append(next_token_id.item())

In [75]:
tokenizer.decode(input_ids)


'It is also helpful to choose learning goals that are meaningful rather than following every new trend. People learn more effectively when'